In [0]:
from pyspark.sql import SparkSession
# ----- To create the spark session
spark = SparkSession.builder.appName('sales_data').getOrCreate()

***Define explicite Schema***

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType

schema = StructType([
    StructField("product_id",IntegerType(),True),
    StructField("customer_id",StringType(),True),
    StructField("order_date",DateType(),True),
    StructField("location",StringType(),True),
    StructField("source_order",StringType(),True)
])


***Read the data***

In [0]:
sales_df = spark.read.format('csv').option('Inferschema',True).schema(schema).option('headers',True).option('multiline',True).load('/Volumes/pyspark/default/data/sales.csv.txt')
sales_df.display()

product_id,customer_id,order_date,location,source_order
1,A,2023-01-01,India,Swiggy
2,A,2022-01-01,India,Swiggy
2,A,2023-01-07,India,Swiggy
3,A,2023-01-10,India,Restaurant
3,A,2022-01-11,India,Swiggy
3,A,2023-01-11,India,Restaurant
2,B,2022-02-01,India,Swiggy
2,B,2023-01-02,India,Swiggy
1,B,2023-01-04,India,Restaurant
1,B,2023-02-11,India,Swiggy


***adding year, month,quater***

In [0]:
from pyspark.sql.functions import month, year,quarter
sales_df = sales_df.withColumn("order_month",month(sales_df.order_date))
sales_df = sales_df.withColumn("order_year",year(sales_df.order_date))
sales_df = sales_df.withColumn("order_quarter",quarter(sales_df.order_date))
sales_df.display()

product_id,customer_id,order_date,location,source_order,order_month,order_quarter,order_year
1,A,2023-01-01,India,Swiggy,1,1,2023
2,A,2022-01-01,India,Swiggy,1,1,2022
2,A,2023-01-07,India,Swiggy,1,1,2023
3,A,2023-01-10,India,Restaurant,1,1,2023
3,A,2022-01-11,India,Swiggy,1,1,2022
3,A,2023-01-11,India,Restaurant,1,1,2023
2,B,2022-02-01,India,Swiggy,2,1,2022
2,B,2023-01-02,India,Swiggy,1,1,2023
1,B,2023-01-04,India,Restaurant,1,1,2023
1,B,2023-02-11,India,Swiggy,2,1,2023


***Read menu data***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
schema = StructType([
    StructField("product_id",IntegerType(),True),
        StructField("product_name",IntegerType(),True),
        StructField("price",StringType(),True)
])

In [0]:
menu_df = spark.read.format('csv').option("schema",True).schema(schema).load("/Volumes/pyspark/default/data/menu.csv.txt")

**Total amount spend by each customer**

In [0]:
total_amount_spent = (sales_df.join(menu_df,'product_id').groupBy('customer_id').agg(sum('price')).orderBy('customer_id'))
display(total_amount_spent)

customer_id,sum(price)
A,4260.0
B,4440.0
C,2400.0
D,1200.0
E,2040.0


***total amount spent by each food category***


In [0]:
total_amount_spent = (sales_df.join(menu_df,'product_id').groupBy('product_name').agg(sum('price')).orderBy(col('product_name').desc()))
display(total_amount_spent)

product_name,sum(price)
null,14340.0


**Total Amount of sales in each month**

In [0]:
df1 = (sales_df.join(menu_df,'product_id').groupBy('order_month').agg(sum('price')).orderBy(col('order_month').desc()))
df1.display()

order_month,sum(price)
11,910.0
7,910.0
6,2960.0
5,2960.0
3,910.0
2,2730.0
1,2960.0


**Yearly sales**

In [0]:
from pyspark.sql.functions import year
df2 = (sales_df.join(menu_df,'product_id').groupBy('order_year').agg(sum('price')).orderBy('order_year'))
df2.display()

order_year,sum(price)
2022,4350.0
2023,9990.0


**quaterly sales**

In [0]:
df3 = (sales_df.join(menu_df,'product_id').groupBy('order_quarter').agg({'price':'sum'})
       .orderBy('order_quarter'))                  
display(df3)

order_quarter,sum(price)
1,6600.0
2,5920.0
3,910.0
4,910.0


**how many times each product purchased**

In [0]:
most_df = (sales_df.join(menu_df,'product_id').groupBy('product_id','product_name').agg(count('product_id').alias('product_count')).orderBy('product_count',ascending=[0,0]).drop('product_id'))
most_df.display()

product_name,product_count
null,48
null,24
null,21
null,12
null,6
null,6


**Top order Item**

In [0]:
from pyspark.sql.functions import *
most_df = (sales_df.join(menu_df,'product_id').groupBy('product_id','product_name')
           .agg(count('product_id').alias('product_count')).
           orderBy('product_count',ascending=0)
           .drop('product_id').limit(1)

           )
display(most_df)

product_name,product_count
null,48


***frequency of customer visited to restaurant***

In [0]:
from pyspark.sql.functions import countDistinct

df = sales_df.filter(col('source_order')=='Restaurant')
df = df.groupBy('customer_id').agg(countDistinct('order_date'))
df.display()

customer_id,count(DISTINCT order_date)
B,6
A,6
E,5
D,1
C,3


**Total sales by each country**

In [0]:
country_sales = (sales_df.join(menu_df,'product_id').groupBy('location').agg(count('price')))
country_sales.display()

location,count(price)
USA,21
India,39
UK,57


**total sales by order_source**

In [0]:
order_source_df = (sales_df.join(menu_df,'product_id').groupBy('source_order').agg(sum('price')))
order_source_df.display()

source_order,sum(price)
zomato,4920.0
Restaurant,3090.0
Swiggy,6330.0
